# Predictive performance across OpenML-CC18

**Hypothesis:** Forest Sketch with one, two, or three iterations produces a distinguishable held-out accuracy relative to a standard random forest, and the difference is consistent across a multi-dataset OpenML-CC18 sample.

Each `(dataset_id, repetition_id)` pair is one paired block. For every downstream classifier, the analysis compares its raw-input baseline with Forest Sketch at `T=1`, `T=2`, and `T=3` within every dataset/repetition block. The default classifier set contains logistic regression, a random forest, and an RBF SVM with automatic feature scaling and the default regularization convention. The joint-block analysis is exploratory; a confirmatory analysis should also aggregate repetitions within each dataset so that repeated seeds do not artificially increase the apparent number of independent datasets.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import t
from sklearn.exceptions import DataDimensionalityWarning

warnings.filterwarnings('ignore', category=DataDimensionalityWarning)

from _hypothesis_utils import (
    DEFAULT_DIMENSION_RATIO,
    DEFAULT_N_ESTIMATORS,
    classification_score,
    make_datasets,
    openml_classification_split,
    openml_output_classifier,
    forest_sketch,
    plot_critical_difference,
    timed_fit_transform,
)

DATASET_COUNT = 4
MAX_DATASET_SIZE = 2000
DATASET_NAMES = None
# Example: DATASET_NAMES = ['kr-vs-kp', 'letter', 'balance-scale', 'mfeat-factors', 'mfeat-fourier']
DATASETS = make_datasets(
    n=None if DATASET_NAMES is not None else DATASET_COUNT,
    max_size=MAX_DATASET_SIZE,
    seed=42,
    dataset_names=DATASET_NAMES,
)
REPETITIONS = tuple(range(5))
ITERATIONS = (1, 2, 3)
CLASSIFIER_LABELS = {
    'logistic_regression': 'Logistic Regression',
    'random_forest': 'Random Forest',
    'svm_rbf_auto_lambda': 'SVM RBF',
}
# Edit this tuple to select which downstream classifiers to compare.
CLASSIFIERS = tuple(CLASSIFIER_LABELS)
METHODS = tuple(
    [f'{CLASSIFIER_LABELS[classifier]} baseline' for classifier in CLASSIFIERS]
    + [
        f'Forest Sketch T={iterations} ({CLASSIFIER_LABELS[classifier]})'
        for classifier in CLASSIFIERS
        for iterations in ITERATIONS
    ]
)
# Best non-iteration settings from the latest notebook 16 smoke search.
N_ESTIMATORS = 300
DIMENSION_RATIO = 20
DIMENSION_MODE = 'expanding'
OUTPUT_FORMAT = 'sparse'
INITIAL_PROJECTION_TYPE = 'sparse'
PATH_PROJECTION_TYPE = 'sparse'
CONCAT_PROJECTION_TYPE = 'sparse'
NORMALIZATION = 'none'

dataset_summary = pd.DataFrame([
    {
        'dataset_id': dataset.dataset_id,
        'name': dataset.name,
        'n_original': dataset.n_original,
        'n_used': dataset.n_used,
        'n_features': len(dataset.feature_names),
    }
    for dataset in DATASETS
])
display(dataset_summary)

,dataset_id,name,n_original,n_used,n_features
0,3,kr-vs-kp,3196,2000,36
1,6,letter,20000,2000,16
2,11,balance-scale,625,625,4
3,12,mfeat-factors,2000,2000,216


In [ ]:
rows = []
evaluation_jobs = [
    (dataset, repetition_id)
    for dataset in DATASETS
    for repetition_id in REPETITIONS
]
for dataset, repetition_id in tqdm(
    evaluation_jobs, total=len(evaluation_jobs), desc='Evaluation blocks'
):
        X_train, X_test, y_train, y_test, preprocessor = openml_classification_split(
            dataset, seed=repetition_id
        )
        p = X_train.shape[1]
        # The estimator resolves d=DIMENSION_RATIO*p from the encoded width.

        # Score every downstream classifier on the raw input baseline.
        for classifier in CLASSIFIERS:
            accuracy, errors = classification_score(
                openml_output_classifier(
                    classifier, seed=repetition_id,
                    n_estimators=N_ESTIMATORS,
                ),
                X_train, y_train, X_test, y_test,
            )
            rows.append({
                'task_id': dataset.task_id,
                'dataset_id': dataset.dataset_id,
                'dataset': dataset.name,
                'repetition_id': repetition_id,
                'n_original': dataset.n_original,
                'n_used': dataset.n_used,
                'p_encoded': p,
                'dimension': p,
                'iterations': 0,
                'classifier': classifier,
                'method': f'{CLASSIFIER_LABELS[classifier]} baseline',
                'accuracy': accuracy,
                'errors': errors,
            })
        for iterations in ITERATIONS:
            sketch = forest_sketch(
                repetition_id,
                dimension_ratio=DIMENSION_RATIO,
                n_iterations=iterations,
                n_estimators=N_ESTIMATORS,
                dimension_mode=DIMENSION_MODE,
                output_format=OUTPUT_FORMAT,
                initial_projection_type=INITIAL_PROJECTION_TYPE,
                path_projection_type=PATH_PROJECTION_TYPE,
                concat_projection_type=CONCAT_PROJECTION_TYPE,
                normalization=NORMALIZATION,
            )
            X_train_view, fit_wall, fit_cpu = timed_fit_transform(
                sketch, X_train, y_train
            )
            dimension = sketch.n_components_
            X_test_view = sketch.transform(X_test)
            for classifier in CLASSIFIERS:
                accuracy, errors = classification_score(
                    openml_output_classifier(
                        classifier, seed=repetition_id,
                        n_estimators=N_ESTIMATORS,
                    ),
                    X_train_view, y_train, X_test_view, y_test,
                )
                rows.append({
                    'task_id': dataset.task_id,
                    'dataset_id': dataset.dataset_id,
                    'dataset': dataset.name,
                    'repetition_id': repetition_id,
                    'n_original': dataset.n_original,
                    'n_used': dataset.n_used,
                    'p_encoded': p,
                    'dimension': dimension,
                    'iterations': iterations,
                    'classifier': classifier,
                    'method': f'Forest Sketch T={iterations} ({CLASSIFIER_LABELS[classifier]})',
                    'accuracy': accuracy,
                    'errors': errors,
                    'fit_wall_seconds': fit_wall,
                    'fit_cpu_seconds': fit_cpu,
                })

results = pd.DataFrame(rows)
results.sort_values(['dataset_id', 'repetition_id', 'iterations'])

Evaluation blocks:   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
summary = results.groupby(['dataset', 'method'], as_index=False).agg(
    mean_accuracy=('accuracy', 'mean'),
    std_accuracy=('accuracy', 'std'),
)
display(summary)

ACCURACY_THRESHOLD = 0.8
dataset_min_accuracy = summary.groupby('dataset')['mean_accuracy'].min()
dataset_order = summary['dataset'].drop_duplicates().tolist()
high_accuracy_datasets = [
    dataset for dataset in dataset_order
    if dataset_min_accuracy[dataset] > ACCURACY_THRESHOLD
]
low_accuracy_datasets = [
    dataset for dataset in dataset_order
    if dataset_min_accuracy[dataset] <= ACCURACY_THRESHOLD
]

fig, axes = plt.subplots(2, 1, figsize=(10, 10))
panel_specs = [
    ('All methods: minimum mean accuracy > threshold', high_accuracy_datasets),
    ('All remaining datasets', low_accuracy_datasets),
]
legend_handles, legend_labels = [], []
for ax, (panel_title, datasets) in zip(axes, panel_specs):
    ax.set_title(panel_title)
    if not datasets:
        ax.text(0.5, 0.5, 'No datasets', ha='center', va='center')
        ax.set_xticks([])
        continue
    panel_summary = summary[summary['dataset'].isin(datasets)]
    means = panel_summary.pivot(
        index='dataset', columns='method', values='mean_accuracy'
    ).reindex(index=datasets, columns=METHODS)
    stds = panel_summary.pivot(
        index='dataset', columns='method', values='std_accuracy'
    ).reindex(index=datasets, columns=METHODS)
    means.plot(
        kind='bar', yerr=stds, capsize=3, ax=ax, legend=False
    )
    panel_y_min = (means - stds).min().min()
    ax.set_ylim(bottom=panel_y_min)
    if not legend_handles:
        legend_handles, legend_labels = ax.get_legend_handles_labels()
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=90)

axes[0].set_ylabel('Mean held-out accuracy ± std')
axes[1].set_ylabel('Mean held-out accuracy ± std')
fig.legend(
    legend_handles, legend_labels, title='Method', loc='lower center',
    bbox_to_anchor=(0.5, -0.01), ncol=min(3, len(METHODS))
)
fig.suptitle('OpenML-CC18 predictive performance', y=1.02)
fig.tight_layout(rect=(0, 0.10, 1, 0.98))


In [ ]:
plot_critical_difference(
    results,
    block_column=['dataset_id', 'repetition_id'],
    method_column='method',
    score_column='accuracy',
    title='Critical differences over dataset × repetition blocks',
)

paired = results.pivot(
    index=['dataset_id', 'repetition_id'],
    columns='method',
    values='accuracy',
)
overall_gain_rows = []
per_dataset_gain_rows = []
for classifier in CLASSIFIERS:
    baseline_method = f'{CLASSIFIER_LABELS[classifier]} baseline'
    reference = paired[baseline_method]
    for iterations in ITERATIONS:
        method = f'Forest Sketch T={iterations} ({CLASSIFIER_LABELS[classifier]})'
        gain = paired[method] - reference
        half_width = t.ppf(0.975, len(gain) - 1) * gain.std(ddof=1) / np.sqrt(len(gain))
        overall_gain_rows.append({
            'method': method,
            'reference': baseline_method,
            'mean_gain': gain.mean(),
            'ci_low': gain.mean() - half_width,
            'ci_high': gain.mean() + half_width,
            'n_blocks': len(gain),
        })
        dataset_stats = gain.groupby(level=0).agg(['mean', 'std'])
        for dataset_id, stats in dataset_stats.iterrows():
            per_dataset_gain_rows.append({
                'dataset_id': dataset_id,
                'method': method,
                'reference': baseline_method,
                'mean_gain': stats['mean'],
                'std_gain': stats['std'],
            })

overall_gain_summary = pd.DataFrame(overall_gain_rows)
per_dataset_gain_summary = pd.DataFrame(per_dataset_gain_rows)
display(overall_gain_summary.style.format({
    'mean_gain': '{:+.3f}',
    'ci_low': '{:+.3f}',
    'ci_high': '{:+.3f}',
}))
display(per_dataset_gain_summary.style.format({
    'mean_gain': '{:+.3f}',
    'std_gain': '{:.3f}',
}))

The primary evidence is the CD diagram and the paired gain of each Forest Sketch iteration relative to the matching raw-input baseline classifier. A positive overall gain is not sufficient for a cross-dataset claim if it is driven by one dataset; the per-dataset paired gains should be reported as well.